# Data Stats

Computes the summary statistics described in `data/README.md`.

In [147]:
import json
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path(".")  # run from data/
TRANSCRIPTS = DATA_DIR / "transcripts" / "step_up.jsonl"
ANNOTATIONS = DATA_DIR / "teacher_annotations" / "step_up_annotations.jsonl"

## transcripts/step_up.jsonl

In [148]:
transcripts = []
with open(TRANSCRIPTS) as f:
    for line in f:
        line = line.strip()
        if line:
            transcripts.append(json.loads(line))

print(f"Total tutoring sessions: {len(transcripts):,}")

Total tutoring sessions: 31,449


In [149]:
# Sessions per batch
batch_counts = Counter(r["batch"] for r in transcripts)
print("Sessions per batch:")
for batch in sorted(batch_counts):
    print(f"  {batch}: {batch_counts[batch]:,}")

Sessions per batch:
  2025-10-16: 462
  2026-02-13: 109
  2026-03-24: 10,878
  2026-04-27: 10,000
  2026-05-29: 10,000


In [150]:
# Turns per session
turn_counts = [len(r["turns"]) for r in transcripts]
avg_turns = sum(turn_counts) / len(turn_counts)
print(f"Turns per session — avg: {avg_turns:.0f}, min: {min(turn_counts)}, max: {max(turn_counts)}")

Turns per session — avg: 325, min: 3, max: 1789


In [151]:
# Whitespace-tokenized token length per turn
all_turn_lengths = []
for r in transcripts:
    for t in r["turns"]:
        text = t.get("text", "") or ""
        all_turn_lengths.append(len(text.split()))

avg_tok = sum(all_turn_lengths) / len(all_turn_lengths)
print(f"Whitespace-token length per turn — avg: {avg_tok:.1f}, min: {min(all_turn_lengths)}, max: {max(all_turn_lengths)}")
print(f"(across {len(all_turn_lengths):,} turns total)")

Whitespace-token length per turn — avg: 11.5, min: 0, max: 4058
(across 10,232,059 turns total)


In [152]:
# has_video distribution
video_counts = Counter(r["has_video"] for r in transcripts)
print(f"has_video=True:  {video_counts[True]:,} ({100*video_counts[True]/len(transcripts):.1f}%)")
print(f"has_video=False: {video_counts[False]:,} ({100*video_counts[False]/len(transcripts):.1f}%)")

has_video=True:  30,987 (98.5%)
has_video=False: 462 (1.5%)


In [153]:
# Session duration distribution (in minutes)
durations = []
for r in transcripts:
    turns = r["turns"]
    if turns:
        duration_sec = turns[-1]["end_seconds"] - turns[0]["start_seconds"]
        durations.append(duration_sec / 60)

avg_dur = sum(durations) / len(durations)
print(f"Session duration (minutes) — avg: {avg_dur:.1f}, min: {min(durations):.1f}, max: {max(durations):.1f}")

Session duration (minutes) — avg: 52.7, min: 0.9, max: 180.4


In [154]:
# Unique tutors and students
tutor_ids = {r["session"].get("tutor_id") for r in transcripts if r["session"].get("tutor_id")}
student_ids = {r["session"].get("student_id") for r in transcripts if r["session"].get("student_id")}
print(f"Unique tutors:  {len(tutor_ids):,}")
print(f"Unique students: {len(student_ids):,}")

Unique tutors:  1,116
Unique students: 1,302


## teacher_annotations/step_up_annotations.jsonl

In [155]:
annotations = []
with open(ANNOTATIONS) as f:
    for line in f:
        line = line.strip()
        if line:
            annotations.append(json.loads(line))

print(f"Total annotator sessions: {len(annotations):,}")

Total annotator sessions: 1,520


In [156]:
type_counts = Counter(r["annotation_type"] for r in annotations)
print("Annotator sessions per annotation type:")
for atype in sorted(type_counts):
    print(f"  {atype}: {type_counts[atype]}")

Annotator sessions per annotation type:
  caption: 79
  rapport: 801
  scaffolding: 640


In [157]:
ann_batch_counts = Counter(r["batch"] for r in annotations)
print("Annotator sessions per batch:")
for batch in sorted(ann_batch_counts):
    print(f"  {batch}: {ann_batch_counts[batch]}")

Annotator sessions per batch:
  2025-10-16: 1520


In [158]:
# Null turn_number_start/end in turn_annotations, by type
for atype in ("rapport", "scaffolding"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_ta = sum(len(r["turn_annotations"]) for r in records)
    null_ta = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is None and ta.get("turn_number_end") is None
    )
    pct = 100 * null_ta / total_ta if total_ta else 0
    print(f"{atype}: {null_ta} / {total_ta} null-anchored turn_annotations ({pct:.1f}%)")

rapport: 27 / 5653 null-anchored turn_annotations (0.5%)
scaffolding: 42 / 4857 null-anchored turn_annotations (0.9%)


In [159]:
# Cut point coverage by annotation type
for atype in ("scaffolding", "rapport", "caption"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total_moments = sum(len(r["turn_annotations"]) for r in records)
    records_with_cut = sum(
        1 for r in records
        if any("cut_turn" in ta for ta in r["turn_annotations"])
    )
    moments_with_cut = sum(
        1
        for r in records
        for ta in r["turn_annotations"]
        if "cut_turn" in ta
    )
    print(f"{atype}:")
    print(f"  annotator sessions (annotator_id, transcript_id) with any cut_turn: {records_with_cut} / {len(records)}")
    print(f"  key moments (annotator_id, transcript_id, turn_start, turn_end) with cut_turn: {moments_with_cut} / {total_moments}")

    if atype in ("scaffolding", "rapport"):
        spans_with_cut = set()
        spans_total = set()
        for r in records:
            tid = r["transcript_id"]
            for ta in r["turn_annotations"]:
                s, e = ta.get("turn_number_start"), ta.get("turn_number_end")
                if s is None:
                    continue
                span = (tid, s, e)
                spans_total.add(span)
                if "cut_turn" in ta:
                    spans_with_cut.add(span)
        n_total = len(spans_total)
        n_cut = len(spans_with_cut)
        pct = 100 * n_cut / n_total if n_total else 0
        print(f"  unique spans (transcript_id, turn_start, turn_end) with cut_turn: {n_cut} / {n_total} ({pct:.1f}%)")
    print()

scaffolding:
  annotator sessions (annotator_id, transcript_id) with any cut_turn: 462 / 640
  key moments (annotator_id, transcript_id, turn_start, turn_end) with cut_turn: 3737 / 4857
  unique spans (transcript_id, turn_start, turn_end) with cut_turn: 1284 / 1536 (83.6%)

rapport:
  annotator sessions (annotator_id, transcript_id) with any cut_turn: 692 / 801
  key moments (annotator_id, transcript_id, turn_start, turn_end) with cut_turn: 5034 / 5653
  unique spans (transcript_id, turn_start, turn_end) with cut_turn: 928 / 928 (100.0%)

caption:
  annotator sessions (annotator_id, transcript_id) with any cut_turn: 0 / 79
  key moments (annotator_id, transcript_id, turn_start, turn_end) with cut_turn: 0 / 5547



In [160]:
# SAR field whitespace-token lengths (scaffolding + rapport only)
sar_fields = {"situation": [], "action": [], "result": []}
for r in annotations:
    if r["annotation_type"] not in ("scaffolding", "rapport"):
        continue
    for ta in r["turn_annotations"]:
        for field in sar_fields:
            text = ta.get(field) or ""
            sar_fields[field].append(len(text.split()))

print("SAR field whitespace-token lengths (scaffolding + rapport, all turn_annotations):")
for field, lengths in sar_fields.items():
    avg = sum(lengths) / len(lengths)
    print(f"  {field}: avg {avg:.1f}, min {min(lengths)}, max {max(lengths)}")

SAR field whitespace-token lengths (scaffolding + rapport, all turn_annotations):
  situation: avg 24.2, min 1, max 256
  action: avg 19.6, min 1, max 342
  result: avg 29.5, min 1, max 308


In [161]:
# moment_id coverage and mapping
# NOTE: moment_id is only present on a subset of key moments — those identified through
# the structured interface that assigns canonical moment IDs. Moments annotated without
# going through that flow have no moment_id. The unique-span counts below are therefore
# scoped to this subset only; they will be smaller than the total unique spans from cell-15.

for atype in ("scaffolding", "rapport"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    total = sum(len(r["turn_annotations"]) for r in records)
    with_mid = sum(1 for r in records for ta in r["turn_annotations"] if "moment_id" in ta)
    print(f"{atype}: {with_mid} / {total} key moments have a moment_id ({100*with_mid/total:.1f}%)")
print()

# moment_id <-> (transcript_id, turn_start, turn_end) mapping (scoped to moments with moment_id)
moment_to_coords = defaultdict(set)
coords_to_moments = defaultdict(set)

for r in annotations:
    if r.get("annotation_type") not in ("scaffolding", "rapport"):
        continue
    conv_id = r["transcript_id"]
    for ta in r.get("turn_annotations", []):
        if "moment_id" not in ta:
            continue
        mid = ta["moment_id"]
        coords = (conv_id, ta.get("turn_number_start"), ta.get("turn_number_end"))
        moment_to_coords[mid].add(coords)
        coords_to_moments[coords].add(mid)

multi_coord = {mid: c for mid, c in moment_to_coords.items() if len(c) > 1}
multi_moment = {c: m for c, m in coords_to_moments.items() if len(m) > 1}

print(f"Among moments with a moment_id:")
print(f"  Unique moment_ids:                    {len(moment_to_coords)}")
print(f"  Unique (transcript_id, turn_start, turn_end): {len(coords_to_moments)}")
print(f"  moment_ids mapping to >1 coord:       {len(multi_coord)}  (moment_id -> coords is injective)")
print(f"  coords mapping to >1 moment_id:       {len(multi_moment)}  (coords -> moment_id is NOT injective)")
print()
print("Note: unique moment_ids > unique coords because two annotators can independently")
print("annotate the same span, producing distinct moment_ids for the same coordinates.")

scaffolding: 3226 / 4857 key moments have a moment_id (66.4%)
rapport: 4696 / 5653 key moments have a moment_id (83.1%)

Among moments with a moment_id:
  Unique moment_ids:                    2167
  Unique (transcript_id, turn_start, turn_end): 2106
  moment_ids mapping to >1 coord:       0  (moment_id -> coords is injective)
  coords mapping to >1 moment_id:       44  (coords -> moment_id is NOT injective)

Note: unique moment_ids > unique coords because two annotators can independently
annotate the same span, producing distinct moment_ids for the same coordinates.


In [162]:
# Annotated sessions (unique transcript_ids with scaffolding or rapport)
annotated_ids = {r["transcript_id"] for r in annotations if r["annotation_type"] in ("scaffolding", "rapport")}
print(f"Sessions with scaffolding or rapport annotations: {len(annotated_ids)}")

# Moments per type (excluding null-anchored)
for atype in ("scaffolding", "rapport"):
    records = [r for r in annotations if r["annotation_type"] == atype]
    moments = [
        ta
        for r in records
        for ta in r["turn_annotations"]
        if ta.get("turn_number_start") is not None
    ]
    print(f"{atype} moments (anchored): {len(moments)}")

Sessions with scaffolding or rapport annotations: 209
scaffolding moments (anchored): 4815
rapport moments (anchored): 5626


## Train/Test Split Statistics

In [163]:
import sys
sys.path.insert(0, str(Path(".").resolve().parent))  # repo root
from annotator.core.utils import compute_iou

SPLIT_FILE = DATA_DIR / "split.json"
GT_DIR = DATA_DIR / "ground_truth_hybrid"

with open(SPLIT_FILE) as f:
    split = json.load(f)

def load_split_moments(conv_ids):
    moments = []
    for conv_id in conv_ids:
        path = GT_DIR / f"{conv_id}.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for m in data.get("key_moments", []):
            moments.append({**m, "conversation_id": conv_id})
    return moments

train_moments = load_split_moments(split["train"])
test_moments = load_split_moments(split["test"])
print(f"Train: {len(split['train'])} conversations, {len(train_moments)} moments")
print(f"Test:  {len(split['test'])} conversations, {len(test_moments)} moments")

Train: 102 conversations, 5423 moments
Test:  105 conversations, 5010 moments


In [164]:
def split_stats(moments, conv_ids, label):
    total = len(moments)
    print(f"{'='*50}")
    print(f"{label}  ({len(conv_ids)} conversations, {total} moments)")
    print(f"{'='*50}")

    # Annotation type breakdown with per-type label distribution
    type_counts = Counter(m["annotation_type"] for m in moments)
    print("\nAnnotation type and strategy label distribution:")
    for atype in ("scaffolding", "rapport"):
        n = type_counts[atype]
        tc = Counter(m["strategy_label"] for m in moments if m["annotation_type"] == atype)
        print(f"  {atype}: {n}")
        for lbl in ("effective", "partial", "ineffective"):
            print(f"    {lbl}: {tc[lbl]} ({100*tc[lbl]/n:.1f}%)")

    # Count moments with exact turn_start/turn_end match from a different annotator
    # within the same conversation and annotation type.
    by_conv = defaultdict(list)
    for i, m in enumerate(moments):
        by_conv[m["conversation_id"]].append((i, m))

    exact_match_indices = set()
    for conv_moments in by_conv.values():
        for idx_a, (gi_a, m1) in enumerate(conv_moments):
            for gi_b, m2 in conv_moments[idx_a + 1:]:
                if m1["annotator_id"] == m2["annotator_id"]:
                    continue
                if m1["annotation_type"] != m2["annotation_type"]:
                    continue
                if m1["turn_start"] == m2["turn_start"] and m1["turn_end"] == m2["turn_end"]:
                    exact_match_indices.add(gi_a)
                    exact_match_indices.add(gi_b)

    # Span coverage: unique locations vs. individual records annotating each location
    # n_moments  = individual annotated moments (one per annotator per span)
    # n_spans    = distinct (conv, turn_start, turn_end) locations
    # avg/span   = n_moments / n_spans — how many annotators covered each location on average
    # n_matched  = records that share their exact span with >=1 other annotator
    # NOTE: n_matched counts records, not spans, so it can exceed n_spans.
    #       A span covered by k annotators contributes k matched records.
    print("\nSpan coverage (unique locations vs. annotated moments per location):")
    for atype in ("scaffolding", "rapport"):
        type_moments = [(i, m) for i, m in enumerate(moments) if m["annotation_type"] == atype]
        n_moments = len(type_moments)
        spans = {(m["conversation_id"], m["turn_start"], m["turn_end"]) for _, m in type_moments}
        n_spans = len(spans)
        avg_per_span = n_moments / n_spans if n_spans else 0
        type_indices = {i for i, _ in type_moments}
        n_matched = len(exact_match_indices & type_indices)
        print(f"  {atype}:")
        print(f"    unique spans (distinct locations):        {n_spans}")
        print(f"    annotated moments (one per annotator/span):   {n_moments}  ({avg_per_span:.1f} moments/span on avg)")
        print(f"    annotated moments with a cross-annotator span match: {n_matched} / {n_moments} ({100*n_matched/n_moments:.1f}%)")

split_stats(train_moments, split["train"], "TRAIN")
print()
split_stats(test_moments, split["test"], "TEST")

TRAIN  (102 conversations, 5423 moments)

Annotation type and strategy label distribution:
  scaffolding: 2436
    effective: 1024 (42.0%)
    partial: 468 (19.2%)
    ineffective: 935 (38.4%)
  rapport: 2987
    effective: 1451 (48.6%)
    partial: 693 (23.2%)
    ineffective: 789 (26.4%)

Span coverage (unique locations vs. annotated moments per location):
  scaffolding:
    unique spans (distinct locations):        866
    annotated moments (one per annotator/span):   2436  (2.8 moments/span on avg)
    annotated moments with a cross-annotator span match: 2190 / 2436 (89.9%)
  rapport:
    unique spans (distinct locations):        468
    annotated moments (one per annotator/span):   2987  (6.4 moments/span on avg)
    annotated moments with a cross-annotator span match: 2987 / 2987 (100.0%)

TEST  (105 conversations, 5010 moments)

Annotation type and strategy label distribution:
  scaffolding: 2379
    effective: 1023 (43.0%)
    partial: 431 (18.1%)
    ineffective: 899 (37.8%)
 

## Decomposed Facets (Scaffolding only)

In [165]:
def decomposed_facet_stats(moments, label):
    total = len(moments)
    action_counts = [len(m.get("action_decomposed") or []) for m in moments]
    result_counts = [len(m.get("result_decomposed") or []) for m in moments]
    n_with_action = sum(1 for c in action_counts if c > 0)
    n_with_result = sum(1 for c in result_counts if c > 0)
    print(f"{label} ({total} scaffolding moments)")
    print(f"  Tutor actions (action_decomposed):")
    print(f"    moments with >=1 facet: {n_with_action} ({100*n_with_action/total:.1f}%)")
    print(f"    avg facets/moment:      {sum(action_counts)/total:.2f}  (max: {max(action_counts)})")
    print(f"  Student indicators (result_decomposed):")
    print(f"    moments with >=1 facet: {n_with_result} ({100*n_with_result/total:.1f}%)")
    print(f"    avg facets/moment:      {sum(result_counts)/total:.2f}  (max: {max(result_counts)})")

def load_scaffolding_moments(conv_ids):
    moments = []
    for conv_id in conv_ids:
        path = GT_DIR / f"{conv_id}.json"
        if not path.exists():
            continue
        with open(path) as f:
            data = json.load(f)
        for m in data.get("key_moments", []):
            if m["annotation_type"] == "scaffolding":
                moments.append(m)
    return moments

all_scaffolding = load_scaffolding_moments(
    [p.stem for p in GT_DIR.glob("*.json")]
)
decomposed_facet_stats(all_scaffolding, "ALL")
print()
decomposed_facet_stats(load_scaffolding_moments(split["train"]), "TRAIN")
print()
decomposed_facet_stats(load_scaffolding_moments(split["test"]), "TEST")

ALL (4815 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 4167 (86.5%)
    avg facets/moment:      1.73  (max: 12)
  Student indicators (result_decomposed):
    moments with >=1 facet: 2691 (55.9%)
    avg facets/moment:      0.85  (max: 6)

TRAIN (2436 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 2122 (87.1%)
    avg facets/moment:      1.75  (max: 12)
  Student indicators (result_decomposed):
    moments with >=1 facet: 1382 (56.7%)
    avg facets/moment:      0.88  (max: 6)

TEST (2379 scaffolding moments)
  Tutor actions (action_decomposed):
    moments with >=1 facet: 2045 (86.0%)
    avg facets/moment:      1.71  (max: 9)
  Student indicators (result_decomposed):
    moments with >=1 facet: 1309 (55.0%)
    avg facets/moment:      0.82  (max: 6)
